# IMPORTS

In [ ]:
import os
import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
import numpy as np

def segment_audio_dataset(
    csv_path: str,
    audio_root: str,
    out_dir: str,
    target_sr: int = 16000,
    clip_duration: int = 3,
    padding_mode: str = "silence"
):
    """
    Segments audio files into fixed-length clips and saves them.

    Parameters
    ----------
    csv_path : str
        Path to CSV containing 'filename' and 'target' columns.
    audio_root : str
        Root directory containing audio files organized by label folders.
    out_dir : str
        Output directory to save segmented clips.
    target_sr : int, default=16000
        Target sampling rate.
    clip_duration : int, default=3
        Duration of each clip in seconds.
    padding_mode : str, default="silence"
        How to pad short clips:
        - "silence": pad with zeros (librosa.fix_length).
        - "repeat": tile audio until length is reached (like pad_random).

    Returns
    -------
    pd.DataFrame
        DataFrame containing segmented filenames and labels.
    """
    os.makedirs(out_dir, exist_ok=True)
    clip_samples = target_sr * clip_duration

    df = pd.read_csv(csv_path)
    segmented_rows = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        file_name = row["filename"]
        label = row["target"]

        folder_path = os.path.join(audio_root, str(label))
        audio_path = os.path.join(folder_path, file_name)

        # Try alternative extensions if missing
        if not os.path.exists(audio_path):
            base = os.path.splitext(file_name)[0]
            found = False
            for ext in ["wav", "mp3", "m4a"]:
                alt = os.path.join(folder_path, f"{base}.{ext}")
                if os.path.exists(alt):
                    audio_path = alt
                    found = True
                    break
            if not found:
                print("MISSING:", file_name)
                continue

        try:
            audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
        except Exception as e:
            print("ERROR LOADING FILE:", audio_path, e)
            continue

        total_samples = len(audio)
        num_segments = total_samples // clip_samples

        if num_segments == 0:
            # Handle short clips
            if padding_mode == "silence":
                padded = librosa.util.fix_length(audio, clip_samples)
            elif padding_mode == "repeat":
                num_repeats = int(clip_samples / total_samples) + 1
                padded = np.tile(audio, num_repeats)[:clip_samples]
            else:
                raise ValueError("padding_mode must be 'silence' or 'repeat'")

            out_name = f"{os.path.splitext(file_name)[0]}_seg0.wav"
            out_path = os.path.join(out_dir, out_name)
            sf.write(out_path, padded, target_sr)
            segmented_rows.append([out_name, label])
        else:
            # Regular segmentation
            for seg_id in range(num_segments):
                start = seg_id * clip_samples
                end = start + clip_samples
                clip = audio[start:end]

                out_name = f"{os.path.splitext(file_name)[0]}_seg{seg_id}.wav"
                out_path = os.path.join(out_dir, out_name)
                sf.write(out_path, clip, target_sr)
                segmented_rows.append([out_name, label])

    seg_df = pd.DataFrame(segmented_rows, columns=["filename", "target"])
    seg_df.to_csv(os.path.join(out_dir, "train_audio_segmented.csv"), index=False)

    return seg_df

In [ ]:
from __future__ import annotations
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
import torchaudio
from torch import Tensor
from tqdm import tqdm
import torch.nn as nn
import random

# --- Copy this function from their code ---
def pad_random(x: np.ndarray, max_len: int = 64600):
    """
    Pads or crops a 1D numpy array to a fixed length.
    - If x_len >= max_len: returns a random crop of max_len.
    - If x_len < max_len: tiles (repeats) x until it reaches max_len.
    """
    x_len = x.shape[0]
    
    # if duration is already long enough
    if x_len >= max_len:
        # Take a random segment
        stt = np.random.randint(x_len - max_len + 1) # +1 to include the last possible start
        return x[stt:stt + max_len]

    # if too short, tile (repeat)
    num_repeats = int(max_len / x_len) + 1
    padded_x = np.tile(x, (num_repeats))[:max_len]
    return padded_x

# --- 1. Custom Transform: Gaussian Noise ---
class AddGaussianNoise(object):
    def __init__(self, min_snr_db=10, max_snr_db=30, p=0.5):
        self.min_snr_db = min_snr_db
        self.max_snr_db = max_snr_db
        self.p = p

    def __call__(self, waveform):
        if random.random() > self.p:
            return waveform
        
        signal_power = waveform.pow(2).mean()
        if signal_power == 0:
            return waveform

        snr_db = random.uniform(self.min_snr_db, self.max_snr_db)
        snr = 10 ** (snr_db / 10)
        
        noise_power = signal_power / snr
        noise = torch.randn_like(waveform) * torch.sqrt(noise_power)
        return waveform + noise

# --- 2. Custom Transform: Speed Perturbation ---
class SpeedPerturb(object):
    def __init__(self, sample_rate, p=0.5):
        self.sample_rate = sample_rate
        self.p = p
        # Speed factors: 0.9x (slower/deeper) to 1.1x (faster/higher)
        self.speeds = [0.9, 1.0, 1.1] 

    def __call__(self, waveform):
        if random.random() > self.p:
            return waveform
        
        speed = random.choice(self.speeds)
        if speed == 1.0:
            return waveform

        # Resample acts as speed perturbation (changes duration + pitch)
        new_sr = int(self.sample_rate * speed)
        resampler = torchaudio.transforms.Resample(orig_freq=self.sample_rate, new_freq=new_sr)
        return resampler(waveform)

# --- 3. The Full Robust Dataset ---
class SpeakerDataset(Dataset):
    def __init__(self, csv_path: str, audio_dir: str, label_map: dict = None, eval_mode=False):
        self.df = pd.read_csv(csv_path)
        self.audio_dir = audio_dir
        self.eval_mode = eval_mode
        self.sample_rate = 16000
        self.n_mels = 80
        self.target_len = 48000  # 3 seconds * 16000 Hz

        # --- FEATURE EXTRACTION ---
        self.compute_mfcc = torchaudio.transforms.MFCC(
            sample_rate=self.sample_rate,
            n_mfcc=self.n_mels,
            melkwargs={"n_fft": 400, "win_length": 400, "hop_length": 160, "n_mels": 80}
        )

        # --- AUGMENTATIONS (Waveform Level) ---
        self.add_noise = AddGaussianNoise(min_snr_db=15, max_snr_db=35, p=0.5)
        self.speed_perturb = SpeedPerturb(self.sample_rate, p=0.5)

        # --- AUGMENTATIONS (Spectrogram Level) ---
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=15)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)

        # --- NORMALIZATION ---
        # Instance Norm 1D (treats 80 features as channels)
        self.instance_norm = nn.InstanceNorm1d(self.n_mels)

        # --- LABELS ---
        self.label_map = label_map
        if not self.eval_mode and self.label_map is None:
            self.unique_labels = sorted(self.df['target'].unique())
            self.label_map = {lbl: i for i, lbl in enumerate(self.unique_labels)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['filename']
        filepath = os.path.join(self.audio_dir, filename)
        
        # 1. LOAD
        try:
            waveform, sr = torchaudio.load(filepath)
        except:
            # Fallback for bad files
            waveform = torch.zeros(1, self.target_len)
            sr = self.sample_rate

        # 2. RESAMPLE
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
            waveform = resampler(waveform)

        # 3. AUGMENT WAVEFORM (Train Only)
        if not self.eval_mode:
            # A. Speed Perturb (Changes length!)
            waveform = self.speed_perturb(waveform)
            # B. Add Noise
            waveform = self.add_noise(waveform)

        # 4. FIX LENGTH (Crucial after speed perturb)
        # Pad or Crop to exactly 3 seconds
        curr_len = waveform.shape[1]
        if curr_len < self.target_len:
            pad_amt = self.target_len - curr_len
            waveform = torch.nn.functional.pad(waveform, (0, pad_amt))
        elif curr_len > self.target_len:
            start = random.randint(0, curr_len - self.target_len)
            waveform = waveform[:, start:start+self.target_len]

        # 5. COMPUTE MFCC
        mfcc = self.compute_mfcc(waveform) # [1, 80, 301]

        # 6. INSTANCE NORMALIZATION (Always)
        # Normalize the 80 channels to have mean=0, std=1
        mfcc = self.instance_norm(mfcc)

        # 7. SPEC AUGMENT (Train Only)
        if not self.eval_mode:
            mfcc = self.time_mask(mfcc)
            mfcc = self.freq_mask(mfcc)

        mfcc = mfcc.squeeze(0) # Final shape: [80, 301]

        if self.eval_mode:
            return mfcc, filename
        else:
            label_str = row['target']
            label_idx = self.label_map[label_str]
            return mfcc, torch.tensor(label_idx, dtype=torch.long)

In [ ]:
import torch
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import logging
import sys

def setup_logger(log_dir: Path,console_verbosity:str , log_name: str = "training.log",enable_file = True):
    """
    Initializes and configures a logger for training or workflow monitoring.

    Args:
        log_dir (Path): Directory where the log file will be stored. Created if it doesn't exist.
        console_verbosity (str): Verbosity level for console output. Accepts:
            - "none": disables console logging (sets level above CRITICAL)
            - "all": enables full debug-level logging
            - any other value defaults to INFO-level logging
        log_name (str, optional): Name of the log file to create. Defaults to "training.log".
        enable_file (bool, optional): If True, enables logging to file. Defaults to True.

    Returns:
        logging.Logger: Configured logger instance named "Trainer" with attached handlers.
    """

    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / log_name

    logger = logging.getLogger("Trainer")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()

    # Always add file handler (UTF-8 for emoji safety)
    if enable_file:
        file_handler = logging.FileHandler(log_path, encoding="utf-8")
        file_handler.setLevel(logging.DEBUG)
        file_fmt = logging.Formatter("%(asctime)s - %(message)s")
        file_handler.setFormatter(file_fmt)
        logger.addHandler(file_handler)
    
    console_handler = logging.StreamHandler(sys.stdout)
    if console_verbosity == "none":
        console_handler.setLevel(51)
    elif console_verbosity == "all":
        console_handler.setLevel(logging.DEBUG)
    else: console_handler.setLevel(logging.INFO)
    console_fmt = logging.Formatter("%(message)s")
    console_handler.setFormatter(console_fmt)
    logger.addHandler(console_handler)
    
    return logger


def get_report(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    average: str = 'macro',
    target_names: list[str] = None,
):
    """
    Generates and logs/prints a classification report and confusion matrix.
    """
    # Convert lists to tensors if needed
    if isinstance(y_true, list): y_true = torch.tensor(y_true)
    if isinstance(y_pred, list): y_pred = torch.tensor(y_pred)

    if not target_names:
        num_class = len(torch.unique(y_true))
        target_names = [str(i) for i in range(num_class)]

    f1 = f1_score(y_true, y_pred, zero_division=0, average=average)
    report = classification_report(y_true, y_pred, target_names=target_names, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    msg = (
        f"\n--- Validation Report ---\n"
        #f"F1 Score ({average}): {f1:.4f}\n\n"
        f"Classification Report:\n{report}\n"
        f"Confusion Matrix:\n{cm}\n"
        f"-------------------------\n"
    )

    return f1, msg

In [3]:
!pip install speechbrain

In [4]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import torch.nn as nn
from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN
import math
import torch.nn.functional as F
from tqdm import tqdm
import torch.optim as optim
from pathlib import Path

# Set up dataloader and device

In [5]:
device1 = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device:{device1}")

Using device:cuda:0


In [ ]:
DATASET_ROOT = r"/kaggle/input/comsys-hackathon-6-task-C-3-0-semi-supervised-10/Comsys Task C dataset/Comsys Task C dataset"
original_CSV_path = r"/kaggle/input/comsys-hackathon-6-task-C-3-0-semi-supervised-10/train_audio.csv"
NEW_OUTPUT_DIR = r"/kaggle/working/segmented_train"

In [ ]:
seg_df = segment_audio_dataset(csv_path=original_CSV_path,
    audio_root=DATASET_ROOT,
    out_dir=NEW_OUTPUT_DIR,
    padding_mode="repeat"
)

print(seg_df.head())

In [ ]:
import shutil
shutil.move(r"/kaggle/working/segmented_train/train_audio_segmented.csv",r"/kaggle/working")

In [6]:
# Paths
SEGMENTED_DIR = r"/kaggle/working/segmented_train"
SEGMENTED_CSV = r"/kaggle/working/train_audio_segmented.csv"

# 1. Load the full segmented list
train_df = pd.read_csv(SEGMENTED_CSV)

In [7]:
train_subset, val_subset = train_test_split(
    train_df, 
    test_size=0.15, 
    stratify=train_df['target'],
    random_state=42
)

In [8]:

train_ds = SpeakerDataset(csv_path=SEGMENTED_CSV, audio_dir=SEGMENTED_DIR)
val_ds = SpeakerDataset(csv_path=SEGMENTED_CSV, audio_dir=SEGMENTED_DIR)

# 7. Create Loaders
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4)

print(f"Training Samples: {len(train_ds)}")
print(f"Validation Samples: {len(val_ds)}")

Training Samples: 2144
Validation Samples: 2144


In [9]:
# 3. Check shape inside loop
for feats, labels in train_loader:
    print(feats.shape)  # Should be [32, 80, 301]
    print(labels.shape) # Should be [32]
    break

torch.Size([32, 80, 301])
torch.Size([32])


# Model and AAM Loss

## ECCAPA-TDNN Model

In [10]:
class ECAPA_Classifier(nn.Module):
    def __init__(self, num_speakers, C=512):
        super().__init__()
        
        # 1. Input Normalization (Safety Net)
        # Ensures robustness even if Dataset class misses it
        self.spec_norm = nn.InstanceNorm1d(80) 
        
        # 2. The Backbone
        self.backbone = ECAPA_TDNN(
            input_size=80,             
            channels=[C, C, C, C, 3*C], 
            kernel_sizes=[5, 3, 3, 3, 1],
            dilations=[1, 2, 3, 4, 1],
            attention_channels=128,    
            lin_neurons=192            
        )
        self.embedding_dim = 192

    def forward(self, x):
        # Input x: [Batch, 80, Time]
        
        # Apply Instance Norm
        x = self.spec_norm(x)
        
        # Permute for SpeechBrain [Batch, Time, Feats]
        x = x.permute(0, 2, 1) 
        
        # Get embeddings
        embeddings = self.backbone(x) 
        embeddings = embeddings.squeeze(1) 
        
        return embeddings

## AAM Softmax

In [11]:
class AAMSoftmax(nn.Module):
    def __init__(self, in_features, out_features, scale=30.0, margin=0.2):
        super(AAMSoftmax, self).__init__()
        self.scale = scale
        self.margin = margin
        self.in_features = in_features
        self.out_features = out_features
        
        # The weights W for the classification
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, label):
        # 1. Normalize Inputs and Weights (Cosine Similarity)
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
        # 2. Create the target margin
        # We only apply margin to the *correct* class
        phi = cosine - self.margin
        
        # 3. Create One-hot encoding
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        
        # 4. Apply margin only to ground truth classes
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.scale
        
        return output

# Training

###      Training config

In [12]:
BATCH_SIZE = 32 # Paper uses 128, but 32 is safer for Hackathon GPUs [cite: 165]
LR = 0.001      # Triangular policy starts low, but 1e-3 is good standard [cite: 161]
EPOCHS = 20

In [13]:
num_speakers = 32

model = ECAPA_Classifier(num_speakers=num_speakers).to(device1)

# 1. The AAM Layer (Calculates angular margins)
# We will call this 'margin_layer' instead of 'criterion' to avoid confusion
margin_layer = AAMSoftmax(in_features=192, out_features=num_speakers).to(device1)

# 2. The Actual Loss Function (Calculates error)
# This was missing!
ce_loss = nn.CrossEntropyLoss().to(device1)

# Optimizer: Paper uses Adam [cite: 161]
# Note: We must optimize BOTH the model weights AND the AAMSoftmax weights
# 3. Optimizer
# You must optimize parameters for BOTH the model and the margin layer
optimizer = optim.Adam(
    list(model.parameters()) + list(margin_layer.parameters()), 
    lr=LR, 
    weight_decay=2e-5
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

### Training Loop

In [14]:
# --- Setup logger ---
log_dir = Path("./logs")
logger = setup_logger(log_dir, console_verbosity="all", log_name="supervised_training.log", enable_file=True)

In [15]:


best_val_f1 = 0.0 # To track best model

for epoch in range(EPOCHS):
    # ================= TRAIN PHASE =================
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for features, labels in pbar:
        features, labels = features.to(device1), labels.to(device1)

        optimizer.zero_grad()
        embeddings = model(features)

        logits = margin_layer(embeddings,labels)

        loss = ce_loss(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Monitor accuracy using Cosine approximation
        with torch.no_grad():
            cosine = F.linear(F.normalize(embeddings), F.normalize(margin_layer.weight))
            preds = cosine.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
        acc = (torch.tensor(all_preds) == torch.tensor(all_labels)).float().mean().item()
        pbar.set_postfix({'loss': running_loss/len(train_loader), 'acc': acc})

    # Step Scheduler at the END of epoch
    scheduler.step()

    # Log Train Metrics
    train_f1, msg = get_report(all_labels, all_preds, average="macro")
    logger.info(f"\nEpoch {epoch+1} Summary:")
    logger.info(f"Train Loss: {running_loss/len(train_loader):.4f} | Train F1: {train_f1:.4f}")
    logger.info(msg)

    # ================= VALIDATION PHASE =================
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels = []
    
    # We do not need gradients for validation
    with torch.no_grad():
        for features, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            features, labels = features.to(device1), labels.to(device1)
            
            # Forward
            embeddings = model(features)
            
            # Validation Loss (AAM-Softmax)
            # Note: AAM-Softmax behaves slightly differently in eval, 
            # but calculating it here gives us a good proxy for convergence.
            logits = margin_layer(embeddings,labels)
            v_loss = ce_loss(logits, labels)
            val_loss += v_loss.item()
            
            # Validation Accuracy (Cosine Similarity)
            cosine = F.linear(F.normalize(embeddings), F.normalize(margin_layer.weight))
            preds = cosine.argmax(dim=1)
            
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    # Validation Metrics
    val_f1, val_report = get_report(val_labels, val_preds, average="macro")
    avg_val_loss = val_loss / len(val_loader)
    
    logger.info(val_report)
    logger.info(f"Val Loss: {avg_val_loss:.4f} | Val F1: {val_f1:.4f}")
    
    # ================= CHECKPOINTING =================
    # Save the best model based on F1 Score
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        
        # Save Model
        torch.save(model.state_dict(), f"ecapa_best_model.pth")
        
        # Save Criterion (CRITICAL!)
        torch.save(margin_layer.state_dict(), f"ecapa_best_criterion.pth")
        
        logger.info(f"🔥 New Best Model & Criterion Saved! at epoch{epoch+1} F1: {best_val_f1:.4f}")
    
    # Save regular epoch checkpoint
    if (epoch + 1) % 2 == 0:
        torch.save(model.state_dict(), f"ecapa_epoch{epoch+1}.pth")
        torch.save(margin_layer.state_dict(), f"ecapa_criterion_epoch{epoch+1}.pth")
        

Epoch 1/20 [Train]: 100%|██████████| 67/67 [00:13<00:00,  5.01it/s, loss=5.43, acc=0.633] 


Epoch 1 Summary:


Train Loss: 5.4297 | Train F1: 0.6110

--- Validation Report ---
Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.62      0.64        61
           1       0.47      0.37      0.41        62
           2       0.41      0.44      0.42        70
           3       0.65      0.60      0.63        60
           4       0.65      0.71      0.68        76
           5       0.74      0.84      0.79        83
           6       0.58      0.57      0.58        63
           7       0.75      0.71      0.73        63
           8       0.60      0.66      0.63        62
           9       0.56      0.35      0.43        43
          10       0.75      0.85      0.80        54
          11       0.57      0.57      0.57        69
          12       0.47      0.49      0.48        57
          13       0.52      0.46      0.49        28
          14       0.74      0.66      0.70        65
          15       0.62      0.58      0.60    

Epoch 1/20 [Val]: 100%|██████████| 67/67 [00:11<00:00,  5.92it/s]


--- Validation Report ---
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.80      0.88        61
           1       0.94      0.50      0.65        62
           2       0.67      0.29      0.40        70
           3       0.87      0.90      0.89        60
           4       0.97      0.76      0.85        76
           5       0.96      0.94      0.95        83
           6       0.72      0.94      0.81        63
           7       0.78      0.84      0.81        63
           8       0.98      0.71      0.82        62
           9       0.75      0.88      0.81        43
          10       0.85      0.94      0.89        54
          11       0.54      0.88      0.67        69
          12       0.86      0.67      0.75        57
          13       0.84      0.75      0.79        28
          14       0.91      0.75      0.82        65
          15       0.84      0.84      0.84        73
          16       0.54      0.

...


--- Validation Report ---
Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        61
           1       0.81      0.74      0.77        62
           2       0.80      0.86      0.83        70
           3       0.98      1.00      0.99        60
           4       1.00      1.00      1.00        76
           5       1.00      1.00      1.00        83
           6       1.00      0.98      0.99        63
           7       1.00      1.00      1.00        63
           8       1.00      1.00      1.00        62
           9       1.00      1.00      1.00        43
          10       1.00      1.00      1.00        54
          11       1.00      1.00      1.00        69
          12       1.00      1.00      1.00        57
          13       1.00      1.00      1.00        28
          14       1.00      1.00      1.00        65
          15       1.00      1.00      1.00        73
          16       1.00      1.


Epoch 20/20 [Train]: 100%|██████████| 67/67 [00:13<00:00,  5.06it/s, loss=0.305, acc=0.982] 


Epoch 20 Summary:
Train Loss: 0.3045 | Train F1: 0.9816

--- Validation Report ---
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        61
           1       0.75      0.82      0.78        62
           2       0.83      0.74      0.78        70
           3       0.98      1.00      0.99        60
           4       0.99      1.00      0.99        76
           5       1.00      1.00      1.00        83
           6       1.00      1.00      1.00        63
           7       1.00      1.00      1.00        63
           8       0.98      1.00      0.99        62
           9       1.00      1.00      1.00        43
          10       1.00      1.00      1.00        54
          11       0.99      0.97      0.98        69
          12       1.00      1.00      1.00        57
          13       0.97      1.00      0.98        28
          14       0.98      1.00      0.99        65
          15       1.00     


Epoch 20/20 [Val]: 100%|██████████| 67/67 [00:10<00:00,  6.11it/s]


--- Validation Report ---
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        61
           1       0.82      0.81      0.81        62
           2       0.84      0.84      0.84        70
           3       1.00      1.00      1.00        60
           4       0.99      1.00      0.99        76
           5       1.00      1.00      1.00        83
           6       1.00      1.00      1.00        63
           7       1.00      1.00      1.00        63
           8       1.00      1.00      1.00        62
           9       1.00      0.98      0.99        43
          10       1.00      1.00      1.00        54
          11       1.00      1.00      1.00        69
          12       0.98      1.00      0.99        57
          13       1.00      1.00      1.00        28
          14       0.98      1.00      0.99        65
          15       1.00      1.00      1.00        73
          16       1.00      0.

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import os
import glob
import torch.nn as nn
import torchaudio
from tqdm import tqdm
from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN

# --- CONFIG ---
TEST_AUDIO_DIR = r"/kaggle/input/comsys-hackathon-6-task-b-3-0-supervised/NewUpdatedTest_All_Task3" 
OUTPUT_CSV = "supervised_submission_robust.csv"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Update Paths to your NEW artifacts
# You identified epoch 18 as the best. Update these paths to where you saved them.
MODEL_PATH = "/kaggle/working/ecapa_best_model.pth" 
CRITERION_PATH = "/kaggle/working/ecapa_best_criterion.pth" # Or whichever criterion corresponds to epoch 18

# --- 2. Define The EXACT Model Architecture Used in Training ---
class ECAPA_Classifier(nn.Module):
    def __init__(self, num_speakers, C=512):
        super().__init__()
        
        # CRITICAL: This was added in your robust training loop
        self.spec_norm = nn.InstanceNorm1d(80) 
        
        self.backbone = ECAPA_TDNN(
            input_size=80,             
            channels=[C, C, C, C, 3*C], 
            kernel_sizes=[5, 3, 3, 3, 1],
            dilations=[1, 2, 3, 4, 1],
            attention_channels=128,    
            lin_neurons=192            
        )
        self.embedding_dim = 192

    def forward(self, x):
        # Input x: [Batch, 80, Time]
        x = self.spec_norm(x)      # Normalize
        x = x.permute(0, 2, 1)     # Permute for SpeechBrain
        embeddings = self.backbone(x) 
        embeddings = embeddings.squeeze(1) 
        return embeddings

# --- 3. Define The AAM Softmax Layer ---
class AAMSoftmax(nn.Module):
    def __init__(self, in_features, out_features, scale=30.0, margin=0.2):
        super(AAMSoftmax, self).__init__()
        self.scale = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))

# --- 4. Robust Test Dataset ---
class TestDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir):
        self.audio_dir = audio_dir
        self.files = glob.glob(os.path.join(audio_dir, "**/*.wav"), recursive=True)
        if len(self.files) == 0:
            self.files = glob.glob(os.path.join(audio_dir, "**/*.mp3"), recursive=True)
            
        print(f"Found {len(self.files)} test files.")
        
        self.sample_rate = 16000
        self.compute_mfcc = torchaudio.transforms.MFCC(
            sample_rate=16000, n_mfcc=80,
            melkwargs={"n_fft": 400, "win_length": 400, "hop_length": 160, "n_mels": 80}
        )
        
        # Note: We do NOT include 'InstanceNorm' here because we moved it 
        # inside the Model class (self.spec_norm) for safety.

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filepath = self.files[idx]
        filename = os.path.basename(filepath)
        
        try:
            waveform, sr = torchaudio.load(filepath)
        except:
            # Silence fallback
            waveform = torch.zeros(1, 48000)
            sr = 16000

        if sr != 16000:
            resampler = torchaudio.transforms.Resample(sr, 16000)
            waveform = resampler(waveform)
            
        # Fix Length (3 seconds / 48000 samples)
        max_len = 48000
        if waveform.shape[1] < max_len:
            waveform = F.pad(waveform, (0, max_len - waveform.shape[1]))
        else:
            # Center crop for inference usually works better than random
            center = waveform.shape[1] // 2
            start = center - (max_len // 2)
            waveform = waveform[:, start:start+max_len]
            
        mfcc = self.compute_mfcc(waveform).squeeze(0) # [80, Time]
        return mfcc, filename

# --- 5. Execution ---
test_ds = TestDataset(TEST_AUDIO_DIR)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Reconstruct Label Map
TRAIN_CSV_PATH = r"/kaggle/working/train_audio_segmented.csv" # Path to CSV used for training
df_train = pd.read_csv(TRAIN_CSV_PATH)
unique_labels = sorted(df_train['target'].unique())
MASTER_LABEL_MAP = {lbl: i for i, lbl in enumerate(unique_labels)}
REVERSE_MAP = {v: k for k, v in MASTER_LABEL_MAP.items()}

# Load Model
model = ECAPA_Classifier(num_speakers=len(unique_labels)).to(DEVICE)
criterion = AAMSoftmax(in_features=192, out_features=len(unique_labels)).to(DEVICE)

print(f"Loading weights from {MODEL_PATH}...")
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
try:
    criterion.load_state_dict(torch.load(CRITERION_PATH, map_location=DEVICE))
except:
    print("Warning: Criterion weights not found. Using raw embeddings (less accurate).")

model.eval()

predictions = []

print("Starting Inference...")
with torch.no_grad():
    for features, filenames in tqdm(test_loader):
        features = features.to(DEVICE)
        
        # Forward
        embeddings = model(features)
        
        # Cosine Similarity against Class Centers (Criterion Weights)
        # This is robust because it compares the test embedding to the "Ideal" speaker center
        cosine = F.linear(F.normalize(embeddings), F.normalize(criterion.weight))
        preds = cosine.argmax(dim=1)
        
        for i in range(len(filenames)):
            pred_id = preds[i].item()
            pred_label = REVERSE_MAP[pred_id]
            predictions.append({"filename": filenames[i], "target": pred_label})

# Save
sub_df = pd.DataFrame(predictions)
sub_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Submission saved to {OUTPUT_CSV}")
print(sub_df.head())